In [ ]:
# pip install ipympl

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy as sp
from scipy.integrate import odeint
import h5py
import math

from mpl_toolkits.mplot3d import axes3d
%matplotlib widget

In [ ]:
# Nc=4
# Nf=1

In [ ]:
Ns=32
Nt=8

In [ ]:
# gstar_HSDM = 2*(Nc**2-1)+2*Nf**2
# gstar_SM = 106.75
# gstar = gstar_SM + gstar_HSDM
# Om_HSDM = gstar_HSDM / (gstar_HSDM+gstar_SM)

In [ ]:
# MP_GeV = 1.220890 * 10**19 # GeV https://physics.nist.gov/cgi-bin/cuu/Value?plkmc2gev

In [ ]:
two_param = False

In [ ]:
class Jackknife:
    def __init__( self, len_data, binsize ):
        self.binsize = binsize
        self.nbins = math.floor( len_data/self.binsize )
        self.N = self.binsize * self.nbins
        self.jack_avg = []
        self.est = 0
        self.var_est = 0

    def set( self, func, list_of_data ):
        for i in range( self.nbins ):
            self.jack_avg.append( func( i, self.binsize, list_of_data ) )

    def do_it( self ):
        for i in range( 0, self.nbins ):
            self.est += self.jack_avg[i]
        self.est /= self.nbins

        for i in range( 0, self.nbins ):
            self.var_est += ( self.jack_avg[i] - self.est )**2
        self.var_est /= self.nbins
        self.var_est *= self.nbins -1

    def do_it_wavg( self, avg ):
        self.est = avg

        for i in range( 0, self.nbins ):
            self.var_est += ( self.jack_avg[i] - self.est )**2
        self.var_est /= self.nbins
        self.var_est *= self.nbins -1

    def mean( self ):
        return self.est

    def var( self ):
        return self.var_est

    def err( self ):
        return np.sqrt(self.var_est)

def simple_mean(i, binsize, np_data):
    resmpld = np.delete(np_data, np.s_[i*binsize:(i+1)*binsize], axis=0)
    return np.mean(resmpld, axis=0)

def jk_avg(i, binsize, np_data):
    return np_data[i]

def format_print(cen, err):
    for i in range(-50, 50):
        if 10**(-i+1)>=err>10**(-i):
            tmp=err*10**(i+1)
            return '{num:.{width}f}'.format(num=cen, width=i+1)+'('+str(round(tmp))+')'

def format_print_w_exact(exact, cen, err):
    for i in range(-50, 50):
        if 10**(-i+1)>=err>10**(-i):
            tmp=err*10**(i+1)
            return '{num:.{width}f}'.format(num=cen, width=i+1)+'('+str(round(tmp))+')'+': '+'{num:.{width}f}'.format(num=(exact-cen)/err, width=i+1)+' sigma'

class Jackknife2:
    def __init__( self, len_data, binsize ):
        self.binsize = binsize
        self.nbins = math.floor( len_data/self.binsize )
        self.N = self.binsize * self.nbins
        self.jack_avg = []
        self.est = 0
        self.cov_est = 0

    def set( self, func, list_of_data ):
        for i in range( self.nbins ):
            self.jack_avg.append( func( i, self.binsize, list_of_data ) )

    def do_it( self ):
        for i in range( 0, self.nbins ):
            self.est += self.jack_avg[i]
        self.est /= self.nbins

        for i in range( 0, self.nbins ):
            self.cov_est += np.kron( self.jack_avg[i] - self.est, self.jack_avg[i] - self.est )
        self.cov_est /= self.nbins
        self.cov_est *= self.nbins -1
        self.cov_est = self.cov_est.reshape( self.est.shape[0], self.est.shape[0] )

    def do_it_wavg( self, avg ):
        # for i in range( 0, self.nbins ):
        #     self.est += self.jack_avg[i]
        # self.est /= self.nbins
        self.est = avg

        for i in range( 0, self.nbins ):
            self.cov_est += np.kron( self.jack_avg[i] - self.est, self.jack_avg[i] - self.est )
        self.cov_est /= self.nbins
        self.cov_est *= self.nbins -1
        self.cov_est = self.cov_est.reshape( self.est.shape[0], self.est.shape[0] )

    def mean( self ):
        return self.est

    def cov( self ):
        return self.cov_est

    def err( self ):
        return np.sqrt(self.cov_est.diagonal())

In [ ]:
font = {'size'   : 12}

colors={
    2:'#006BA4',
    3:'#FF800E',
    4:'#ABABAB',
    "A":'#FFBC79',
    "B":'#5F9ED1',
    "C":'#C85200',
    6:'#898989',
    7:'#A2C8EC',
    8:'#595959',
    9:'#CFCFCF'
}

markers={
    2:"o", 
    3:"d", 
    4:"v", 
    "A":"x", 
    "B":"<", 
    "C":"^", 
}

plt.rcParams.update({
    "text.usetex": True,
    "font.family": "Helvetica",
    'font.size'  : 22
})

In [ ]:
pwd

In [ ]:
nbetasMeas=1000

In [ ]:
# v16 32c windows. S3 window (window0Xn) = the SHOT jk keeplist nodes (m0.4 non-contiguous: gaps
# 767/769/771/774/776); loaded from the clean-room keeplist. DeltaV window (window0X) =
# arange(keeplist_start, ibetac) -- full supercooled range up to Tc (Veff.h5 has all keys; DeltaV->0
# at ibetac, flips beyond -> do NOT trespass ibetac).
ibetac02 = int(np.loadtxt( "../spline_cpp_v16_32c_claude/betac_ibetac_mass0p2000_"+str(Ns)+".dat" )[1])
window02n = np.loadtxt( "../spline_cpp_v16_32c_claude/keeplist_v16_m0p2000_claude.dat" )[:,0].astype(int)
window02 = np.arange( int(window02n[0]), ibetac02 )

ibetac03 = int(np.loadtxt( "../spline_cpp_v16_32c_claude/betac_ibetac_mass0p3000_"+str(Ns)+".dat" )[1])
window03n = np.loadtxt( "../spline_cpp_v16_32c_claude/keeplist_v16_m0p3000_claude.dat" )[:,0].astype(int)
window03 = np.arange( int(window03n[0]), ibetac03 )

ibetac04 = int(np.loadtxt( "../spline_cpp_v16_32c_claude/betac_ibetac_mass0p4000_"+str(Ns)+".dat" )[1])
window04n = np.loadtxt( "../spline_cpp_v16_32c_claude/keeplist_v16_m0p4000_claude.dat" )[:,0].astype(int)
window04 = np.arange( int(window04n[0]), ibetac04 )

In [ ]:
ibetac0s = {
    "0p2000": ibetac02,
    "0p3000": ibetac03,
    "0p4000": ibetac04
}

In [ ]:
windows = {
    "0p2000": window02,
    "0p3000": window03,
    "0p4000": window04
}

In [ ]:
windows2 = {
    "0p2000": window02n,
    "0p3000": window03n,
    "0p4000": window04n
}

In [ ]:
nbetas = {
    '0p2000':9,
    '0p3000':8,
    '0p4000':9
}
nbins=40

In [ ]:
# @@@@ TODO: CHECK FIT WITH PLOTS

In [ ]:
epsilon=0.5
GAM = 2.500 # 3.127   # 3D Ising (6nu-2beta); systematic variation 2.50 (3D tricritical). See gamma_scaling_estimate_claude.md
gamtag = ("c2only_gam%.3f" % GAM).replace('.', 'p').replace('-', 'm')


In [ ]:
from f import F_ainv
from f import F_MB
from f import F_TmTc

In [ ]:
f_ainv = F_ainv( epsilon )
f_MB = F_MB( epsilon )
f_TmTc = F_TmTc( Nt, epsilon )

In [ ]:
# eps=1.0
# eps=0.01
# eps=0.001
# eps=0.0001
# eps=0.00001
# eps=0.

In [ ]:
# def eps_trick(orig, eps):
#     mn = np.mean(orig, axis=0)
#     dev = orig-mn
#     scaled = mn + dev * eps
#     return scaled

In [ ]:
# def eps_trick2(orig, eps, mean):
#     mn = np.mean(orig, axis=0)
#     dev = orig-mn
#     scaled = mn + dev * eps
#     return scaled

# Tc

In [ ]:
# def Tc_MB_fitter( MB, b0, b1, b2 ):
#     return b0 + b1*MB + b2*MB**2

# def dTc_dMB_fitter( MB, b0, b1, b2 ):
#     return b1 + 2.0*b2*MB

def Tc_MB_fitter( MB, b0, b1 ):
    return b0 + b1*MB

def dTc_dMB_fitter( MB, b0, b1 ):
    return b1

In [ ]:
mass="0p2000"
betac_ibetac_m02 = np.loadtxt( "../spline_cpp_v16_32c_claude/betac_ibetac_mass"+mass+"_"+str(Ns)+".dat" )
mass="0p3000"
betac_ibetac_m03 = np.loadtxt( "../spline_cpp_v16_32c_claude/betac_ibetac_mass"+mass+"_"+str(Ns)+".dat" )
mass="0p4000"
betac_ibetac_m04 = np.loadtxt( "../spline_cpp_v16_32c_claude/betac_ibetac_mass"+mass+"_"+str(Ns)+".dat" )

Tc_m0p2000 = f_ainv(betac_ibetac_m02[0], 0.2)/Nt
Tc_m0p3000 = f_ainv(betac_ibetac_m03[0], 0.3)/Nt
Tc_m0p4000 = f_ainv(betac_ibetac_m04[0], 0.4)/Nt

MB_m0p2000 = f_MB(betac_ibetac_m02[0], 0.2)
MB_m0p3000 = f_MB(betac_ibetac_m03[0], 0.3)
MB_m0p4000 = f_MB(betac_ibetac_m04[0], 0.4)

In [ ]:
mass="0p2000"

betas = np.loadtxt( "../spline_cpp_v16_32c_claude/betas_mass"+mass+"_32.dat" )
ibetacs_jk = np.loadtxt( "../spline_cpp_v16_32c_claude/ibetac_jk_mass"+mass+"_"+str(Ns)+".dat" )

betacs_jk = np.array([[betas[ int(ibetacs_jk[jdrop][ibin]) ] for ibin in np.arange(nbins) ] for jdrop in np.arange(nbetas[mass]) ])
Tc_jkdatas_m0p2000 = np.array([[ f_ainv( betacs_jk[jdrop][ibin], 0.2)/Nt for ibin in np.arange(nbins) ] for jdrop in np.arange(nbetas[mass]) ])
MB_jkdatas_m0p2000 = np.array([[ f_MB( betacs_jk[jdrop][ibin], 0.2) for ibin in np.arange(nbins) ] for jdrop in np.arange(nbetas[mass]) ])

In [ ]:
mass="0p3000"

betas = np.loadtxt( "../spline_cpp_v16_32c_claude/betas_mass"+mass+"_32.dat" )
ibetacs_jk = np.loadtxt( "../spline_cpp_v16_32c_claude/ibetac_jk_mass"+mass+"_"+str(Ns)+".dat" )

betacs_jk = np.array([[betas[ int(ibetacs_jk[jdrop][ibin]) ] for ibin in np.arange(nbins) ] for jdrop in np.arange(nbetas[mass]) ])
Tc_jkdatas_m0p3000 = np.array([[ f_ainv( betacs_jk[jdrop][ibin], 0.3)/Nt for ibin in np.arange(nbins) ] for jdrop in np.arange(nbetas[mass]) ])
MB_jkdatas_m0p3000 = np.array([[ f_MB( betacs_jk[jdrop][ibin], 0.3) for ibin in np.arange(nbins) ] for jdrop in np.arange(nbetas[mass]) ])

In [ ]:
mass="0p4000"

betas = np.loadtxt( "../spline_cpp_v16_32c_claude/betas_mass"+mass+"_32.dat" )
ibetacs_jk = np.loadtxt( "../spline_cpp_v16_32c_claude/ibetac_jk_mass"+mass+"_"+str(Ns)+".dat" )

betacs_jk = np.array([[betas[ int(ibetacs_jk[jdrop][ibin]) ] for ibin in np.arange(nbins) ] for jdrop in np.arange(nbetas[mass]) ])
Tc_jkdatas_m0p4000 = np.array([[ f_ainv( betacs_jk[jdrop][ibin], 0.4)/Nt for ibin in np.arange(nbins) ] for jdrop in np.arange(nbetas[mass]) ])
MB_jkdatas_m0p4000 = np.array([[ f_MB( betacs_jk[jdrop][ibin], 0.4) for ibin in np.arange(nbins) ] for jdrop in np.arange(nbetas[mass]) ])

In [ ]:
mass="0p2000"

err_sq = 0.0
for jdrop in np.arange(nbetas[mass]):
    Tc_jkresamp = Tc_jkdatas_m0p2000[jdrop]
    
    jk = Jackknife( nbins, 1 )
    jk.set( jk_avg, Tc_jkresamp )
    
    jk.do_it()
    # jk.do_it_wavg( Tc_m0p2000 )
    err_sq += jk.err()**2
    
delta_Tc_m0p2000 = np.sqrt(err_sq)

In [ ]:
mass="0p3000"

err_sq = 0.0
for jdrop in np.arange(nbetas[mass]):
    Tc_jkresamp = Tc_jkdatas_m0p3000[jdrop]
    
    jk = Jackknife( nbins, 1 )
    jk.set( jk_avg, Tc_jkresamp )
    
    jk.do_it()
    # jk.do_it_wavg( Tc_m0p3000 )
    err_sq += jk.err()**2
    
delta_Tc_m0p3000 = np.sqrt(err_sq)

In [ ]:
mass="0p4000"

err_sq = 0.0
for jdrop in np.arange(nbetas[mass]):
    Tc_jkresamp = Tc_jkdatas_m0p4000[jdrop]
    
    jk = Jackknife( nbins, 1 )
    jk.set( jk_avg, Tc_jkresamp )
    
    jk.do_it()
    # jk.do_it_wavg( Tc_m0p4000 )
    err_sq += jk.err()**2
    
delta_Tc_m0p4000 = np.sqrt(err_sq)

In [ ]:
delta_Tcs = np.array([delta_Tc_m0p2000, delta_Tc_m0p3000, delta_Tc_m0p4000])

In [ ]:
mass="0p2000"

err_sq = 0.0
for jdrop in np.arange(nbetas[mass]):
    MB_jkresamp = MB_jkdatas_m0p2000[jdrop]
    
    jk = Jackknife( nbins, 1 )
    jk.set( jk_avg, MB_jkresamp )
    
    jk.do_it()
    # jk.do_it_wavg( MB_m0p2000 )
    err_sq += jk.err()**2
    
delta_MB_m0p2000 = np.sqrt(err_sq)

In [ ]:
mass="0p3000"

err_sq = 0.0
for jdrop in np.arange(nbetas[mass]):
    MB_jkresamp = MB_jkdatas_m0p3000[jdrop]
    
    jk = Jackknife( nbins, 1 )
    jk.set( jk_avg, MB_jkresamp )
    
    jk.do_it()
    # jk.do_it_wavg( MB_m0p3000 )
    err_sq += jk.err()**2
    
delta_MB_m0p3000 = np.sqrt(err_sq)

In [ ]:
mass="0p4000"

err_sq = 0.0
for jdrop in np.arange(nbetas[mass]):
    MB_jkresamp = MB_jkdatas_m0p4000[jdrop]
    
    jk = Jackknife( nbins, 1 )
    jk.set( jk_avg, MB_jkresamp )
    
    jk.do_it()
    # jk.do_it_wavg( MB_m0p4000 )
    err_sq += jk.err()**2
    
delta_MB_m0p4000 = np.sqrt(err_sq)

In [ ]:
delta_MBs = np.array([delta_MB_m0p2000, delta_MB_m0p3000, delta_MB_m0p4000])

In [ ]:
# fitparams0 = np.array([ 1.210107337539255146e-01, 1.415778904537491438e-02, -5.120170917404638861e-04 ])
fitparams0 = np.array([ 1.210107337539255146e-01, 1.415778904537491438e-02 ])

In [ ]:
Tcs = np.array([Tc_m0p2000, Tc_m0p3000, Tc_m0p4000])
MBs = np.array([MB_m0p2000, MB_m0p3000, MB_m0p4000])

fitparams = fitparams0
for i in range(10):
    d0s = dTc_dMB_fitter( MBs, fitparams[0], fitparams[1] )
    sigma = np.sqrt( delta_Tcs**2 + d0s**2 * delta_MBs**2)
    opt = sp.optimize.curve_fit( f=Tc_MB_fitter, xdata=MBs, ydata=Tcs, 
                                 sigma=sigma,
                              p0=fitparams,
                               full_output=True,
                               absolute_sigma=True)
    fitparams = opt[0]
# for i in range(10):
#     d0s = dTc_dMB_fitter( MBs, fitparams[0], fitparams[1], fitparams[2] )
#     sigma = np.sqrt( delta_Tcs**2 + d0s**2 * delta_MBs**2)
#     opt = sp.optimize.curve_fit( f=Tc_MB_fitter, xdata=MBs, ydata=Tcs, 
#                                  sigma=sigma,
#                               p0=fitparams,
#                                full_output=True,
#                                absolute_sigma=True)

    fitparams = opt[0]
Tcparams=opt[0]

In [ ]:
sigma

In [ ]:
fitparams

In [ ]:
Tcparams

In [ ]:
MBs

In [ ]:
Tcs

In [ ]:
Tc_MB_fitter(MBs, fitparams[0], fitparams[1])

In [ ]:
# yyyy = Tc_MB_fitter(MBs, fitparams[0], fitparams[1], fitparams[2])
yyyy = Tc_MB_fitter(MBs, fitparams[0], fitparams[1] )
plt.clf()
plt.errorbar( MBs, Tcs, 
              yerr=delta_Tcs/epsilon, 
              xerr=delta_MBs/epsilon, 
              ls='none', capsize=3,
            c=colors[2], marker='+', label="${\\rm MC}$" )
plt.plot( MBs, yyyy, c=colors[4], ls='dashed', label='${\\rm fit}$' )
plt.xlabel("$M_B \\sqrt{t_0}$")
plt.ylabel("$T_c \\sqrt{t_0}$")
plt.legend()
plt.savefig("Tc_MB_"+gamtag+".pdf", bbox_inches='tight')
plt.show()

In [ ]:
yyyy = Tc_MB_fitter(MBs, fitparams[0], fitparams[1])
np.sum( (Tcs-yyyy)**2/sigma**2 )/(len(Tcs)-2)*epsilon

In [ ]:
# yyyy = Tc_MB_fitter(MBs, fitparams[0], fitparams[1])
# np.sum( (Tcs-yyyy)**2/sigma**2 )/(len(Tcs)-3)*epsilon

In [ ]:
# vary m=0.2
mass="0p2000"
jk_Tcparams_m0p2000=[]

for jdrop in np.arange(nbetas[mass]):
    fp_jk_=[]
    
    # Tc_jkresamp = eps_trick(Tc_jkdatas_m0p2000[jdrop], eps)
    # MB_jkresamp = eps_trick(MB_jkdatas_m0p2000[jdrop], eps)
    Tc_jkresamp = Tc_jkdatas_m0p2000[jdrop]
    MB_jkresamp = MB_jkdatas_m0p2000[jdrop]
    for ibin in np.arange(nbins):
        Tcs = np.array([Tc_jkresamp[ibin], Tc_m0p3000, Tc_m0p4000])
        MBs = np.array([MB_jkresamp[ibin], MB_m0p3000, MB_m0p4000])

        fitparams = fitparams0
        for i in range(10):
            d0s = dTc_dMB_fitter( MBs, fitparams[0], fitparams[1] )
            sigma = np.sqrt( delta_Tcs**2 + d0s**2 * delta_MBs**2)
            opt = sp.optimize.curve_fit( f=Tc_MB_fitter, xdata=MBs, ydata=Tcs, 
                                         sigma=sigma,
                                      p0=fitparams,
                                       full_output=True,
                                       absolute_sigma=True)
            # d0s = dTc_dMB_fitter( MBs, fitparams[0], fitparams[1], fitparams[2] )
            # sigma = np.sqrt( delta_Tcs**2 + d0s**2 * delta_MBs**2)
            # opt = sp.optimize.curve_fit( f=Tc_MB_fitter, xdata=MBs, ydata=Tcs, 
            #                              sigma=sigma,
            #                           p0=fitparams,
            #                            full_output=True,
            #                            absolute_sigma=True)
        
            fitparams = opt[0]
        fp_jk_.append( opt[0] )

    jk_data = np.array(fp_jk_)
    jk_Tcparams_m0p2000.append( jk_data )

In [ ]:
# vary m=0.3
mass="0p3000"
jk_Tcparams_m0p3000=[]

for jdrop in np.arange(nbetas[mass]):
    fp_jk_=[]
    
    # Tc_jkresamp = eps_trick(Tc_jkdatas_m0p3000[jdrop], eps)
    # MB_jkresamp = eps_trick(MB_jkdatas_m0p3000[jdrop], eps)
    Tc_jkresamp = Tc_jkdatas_m0p3000[jdrop]
    MB_jkresamp = MB_jkdatas_m0p3000[jdrop]
    for ibin in np.arange(nbins):
        Tcs = np.array([Tc_m0p2000, Tc_jkresamp[ibin], Tc_m0p4000])
        MBs = np.array([MB_m0p2000, MB_jkresamp[ibin], MB_m0p4000])

        fitparams = fitparams0
        for i in range(10):
            d0s = dTc_dMB_fitter( MBs, fitparams[0], fitparams[1] )
            sigma = np.sqrt( delta_Tcs**2 + d0s**2 * delta_MBs**2)
            opt = sp.optimize.curve_fit( f=Tc_MB_fitter, xdata=MBs, ydata=Tcs, 
                                         sigma=sigma,
                                      p0=fitparams,
                                       full_output=True,
                                       absolute_sigma=True)
            # d0s = dTc_dMB_fitter( MBs, fitparams[0], fitparams[1], fitparams[2] )
            # sigma = np.sqrt( delta_Tcs**2 + d0s**2 * delta_MBs**2)
            # opt = sp.optimize.curve_fit( f=Tc_MB_fitter, xdata=MBs, ydata=Tcs, 
            #                              sigma=sigma,
            #                           p0=fitparams,
            #                            full_output=True,
            #                            absolute_sigma=True)

        
            fitparams = opt[0]
        fp_jk_.append( opt[0] )

    jk_data = np.array(fp_jk_)
    jk_Tcparams_m0p3000.append( jk_data )

In [ ]:
# vary m=0.4
mass="0p4000"
jk_Tcparams_m0p4000=[]

for jdrop in np.arange(nbetas[mass]):
    fp_jk_=[]
    
    # Tc_jkresamp = eps_trick(Tc_jkdatas_m0p4000[jdrop], eps)
    # MB_jkresamp = eps_trick(MB_jkdatas_m0p4000[jdrop], eps)
    Tc_jkresamp = Tc_jkdatas_m0p4000[jdrop]
    MB_jkresamp = MB_jkdatas_m0p4000[jdrop]  
    for ibin in np.arange(nbins):
        Tcs = np.array([Tc_m0p2000, Tc_m0p3000, Tc_jkresamp[ibin]])
        MBs = np.array([MB_m0p2000, MB_m0p3000, MB_jkresamp[ibin]])

        fitparams = fitparams0
        for i in range(10):
            d0s = dTc_dMB_fitter( MBs, fitparams[0], fitparams[1] )
            sigma = np.sqrt( delta_Tcs**2 + d0s**2 * delta_MBs**2)
            opt = sp.optimize.curve_fit( f=Tc_MB_fitter, xdata=MBs, ydata=Tcs, 
                                         sigma=sigma,
                                      p0=fitparams,
                                       full_output=True,
                                       absolute_sigma=True)
            # d0s = dTc_dMB_fitter( MBs, fitparams[0], fitparams[1], fitparams[2] )
            # sigma = np.sqrt( delta_Tcs**2 + d0s**2 * delta_MBs**2)
            # opt = sp.optimize.curve_fit( f=Tc_MB_fitter, xdata=MBs, ydata=Tcs, 
            #                              sigma=sigma,
            #                           p0=fitparams,
            #                            full_output=True,
            #                            absolute_sigma=True)

        
            fitparams = opt[0]
        fp_jk_.append( opt[0] )

    jk_data = np.array(fp_jk_)
    jk_Tcparams_m0p4000.append( jk_data )

In [ ]:
# class F_Tc:
#     def fitter( self, MB, b0, b1, b2 ):
#         return b0 + b1*MB + b2*MB**2

#     def __init__( self, fp, epsilon ):
#         self.fp = fp

#     def __call__( self, MB ):
#         return self.fitter( MB, self.fp[0], self.fp[1], self.fp[2] )

In [ ]:
# class F_Tc:
#     def fitter( self, MB, b0, b1, b2 ):
#         return b0 + b1*MB # + b2*MB**2

#     def __init__( self, fp, epsilon ):
#         self.fp = fp

#     def __call__( self, MB ):
#         return self.fitter( MB, self.fp[0], self.fp[1] )

In [ ]:
# def Tc_MB_fitter( MB, b0, b1, b2 ):
#     return b0 + b1*MB + b2*MB**2

# def dTc_dMB_fitter( MB, b0, b1, b2 ):
#     return b1 + 2.0*b2*MB

# S3

In [ ]:
def S3hat_fitter(TmTc_MB, Mc, c0, c1, c2, gam ):
    TmTc = TmTc_MB[0]
    MB = TmTc_MB[1]
    return (MB-Mc)**gam  * ( c0 + c1/TmTc + c2/TmTc**2 )

def d0_S3hat_fitter(TmTc_MB, Mc, c0, c1, c2, gam ):
    TmTc = TmTc_MB[0]
    MB = TmTc_MB[1]
    return (MB-Mc)**gam  * ( -1.0*c1/TmTc**2 -2.0 * c2/TmTc**3 )

def d1_S3hat_fitter(TmTc_MB, Mc, c0, c1, c2, gam ):
    TmTc = TmTc_MB[0]
    MB = TmTc_MB[1]
    return gam*(MB-Mc)**(gam-1.0)  * ( c0 + c1/TmTc + c2/TmTc**2 )

In [ ]:
pwd

In [ ]:
mass="0p2000"

S3hats_=[]
for ibeta in windows2[mass]:
    h = h5py.File("../spline_cpp_v16_32c_claude/fit_params_32c_m"+mass+"/S3_"+str(ibeta)+".h5", 
              'r')
    tmp = h['Scl'][()]
    S3hats_.append(tmp)
    h.close()
S3hats_m0p2000 = np.array(S3hats_)

# err_sq = 0.0*windows2[mass]
S3hats_jkdatas_m0p2000 = []
for jdrop in np.arange(nbetas[mass]):

    jk_data_ = []
    for ibin in np.arange(nbins):
        S3hats_ = []
        for ibeta in windows2[mass]:
            print(jdrop, ibin, ibeta)
            h = h5py.File("../spline_cpp_v16_32c_claude/fit_params_32c_m"+mass+"_jk_"+str(jdrop)+"_"+str(nbins)+"_"+str(ibin)+"/S3_"+str(ibeta)+".h5", 
                          'r')
            tmp = h[str(ibeta)+'/Scl'][()]
            h.close()
            # tmp = np.loadtxt("../spline_cpp_v16_32c_claude/fit_params_32c_m"+mass+"_jk_"+str(jdrop)+"_40_"+str(ibin)+"/Scl_"+str(ibeta)+".dat")
            S3hats_.append(tmp)
        
        S3hats = np.array(S3hats_)
        jk_data_.append( S3hats )
    jk_data = np.array( jk_data_ )
    S3hats_jkdatas_m0p2000.append( jk_data )

In [ ]:
mass="0p3000"

S3hats_=[]
for ibeta in windows2[mass]:
    h = h5py.File("../spline_cpp_v16_32c_claude/fit_params_32c_m"+mass+"/S3_"+str(ibeta)+".h5", 
              'r')
    tmp = h['Scl'][()]
    S3hats_.append(tmp)
    h.close()
S3hats_m0p3000 = np.array(S3hats_)

# err_sq = 0.0*windows2[mass]
S3hats_jkdatas_m0p3000 = []
for jdrop in np.arange(nbetas[mass]):

    jk_data_ = []
    for ibin in np.arange(nbins):
        print( jdrop, ibin )
        S3hats_ = []
        for ibeta in windows2[mass]: 
            h = h5py.File("../spline_cpp_v16_32c_claude/fit_params_32c_m"+mass+"_jk_"+str(jdrop)+"_"+str(nbins)+"_"+str(ibin)+"/S3_"+str(ibeta)+".h5", 
                          'r')
            tmp = h[str(ibeta)+'/Scl'][()]
            h.close()
            S3hats_.append(tmp)
        
        S3hats = np.array(S3hats_)
        jk_data_.append( S3hats )
    jk_data = np.array( jk_data_ )
    S3hats_jkdatas_m0p3000.append( jk_data )

In [ ]:
mass="0p4000"

S3hats_=[]
for ibeta in windows2[mass]:
    h = h5py.File("../spline_cpp_v16_32c_claude/fit_params_32c_m"+mass+"/S3_"+str(ibeta)+".h5", 
              'r')
    tmp = h['Scl'][()]
    S3hats_.append(tmp)
    h.close()
S3hats_m0p4000 = np.array(S3hats_)

# err_sq = 0.0*windows2[mass]
S3hats_jkdatas_m0p4000 = []
for jdrop in np.arange(nbetas[mass]):

    jk_data_ = []
    for ibin in np.arange(nbins):
        print( jdrop, ibin )
        S3hats_ = []
        for ibeta in windows2[mass]:
            h = h5py.File("../spline_cpp_v16_32c_claude/fit_params_32c_m"+mass+"_jk_"+str(jdrop)+"_"+str(nbins)+"_"+str(ibin)+"/S3_"+str(ibeta)+".h5", 
                          'r')
            tmp = h[str(ibeta)+'/Scl'][()]
            h.close()
            S3hats_.append(tmp)
        
        S3hats = np.array(S3hats_)
        jk_data_.append( S3hats )
    jk_data = np.array( jk_data_ )
    S3hats_jkdatas_m0p4000.append( jk_data )

In [ ]:
x_dx_y_dy = []

# 0p2
mq = 0.2
mass="0p2000"

betas_all = np.loadtxt( "../spline_cpp_v16_32c_claude/betas_mass"+mass+"_32.dat" )
betac = betas_all[ibetac0s[mass]]
betas = betas_all[windows2[mass]]

MBs = f_MB( betas, mq )
DMBs = np.array([f_MB.D( beta, mq ) for beta in betas ])

dbetas = betas - betac
TmTcs = [f_TmTc( dbeta, mq ) for dbeta in dbetas]
DTmTcs = [f_TmTc.D( dbeta, mq ) for dbeta in dbetas]

x_dx_y_dy.append( [ TmTcs, DTmTcs, MBs, DMBs ] )


# 0p3
mq = 0.3
mass="0p3000"

betas_all = np.loadtxt( "../spline_cpp_v16_32c_claude/betas_mass"+mass+"_32.dat" )
betac = betas_all[ibetac0s[mass]]
betas = betas_all[windows2[mass]]

MBs = f_MB( betas, mq )
DMBs = np.array([f_MB.D( beta, mq ) for beta in betas ])

dbetas = betas - betac
TmTcs = [f_TmTc( dbeta, mq ) for dbeta in dbetas]
DTmTcs = [f_TmTc.D( dbeta, mq ) for dbeta in dbetas]

x_dx_y_dy.append( [ TmTcs, DTmTcs, MBs, DMBs ] )


# 0p4
mq = 0.4
mass="0p4000"

betas_all = np.loadtxt( "../spline_cpp_v16_32c_claude/betas_mass"+mass+"_32.dat" )
betac = betas_all[ibetac0s[mass]]
betas = betas_all[windows2[mass]]

MBs = f_MB( betas, mq )
DMBs = np.array([f_MB.D( beta, mq ) for beta in betas ])

dbetas = betas - betac
TmTcs = [f_TmTc( dbeta, mq ) for dbeta in dbetas]
DTmTcs = [f_TmTc.D( dbeta, mq ) for dbeta in dbetas]

x_dx_y_dy.append( [ TmTcs, DTmTcs, MBs, DMBs ] )

In [ ]:
len(x_dx_y_dy)

In [ ]:
mass="0p2000"

chk=[]

err_sq = 0.0*windows2[mass]
for jdrop in np.arange(nbetas[mass]):
    S3hats_jkresamp = S3hats_jkdatas_m0p2000[jdrop]
    
    jk = Jackknife( nbins, 1 )
    jk.set( jk_avg, S3hats_jkresamp )
    
    jk.do_it()
    # jk.do_it_wavg( S3hats_m0p2000 )
    err_sq += jk.err()**2
    chk.append(jk.mean())
    
delta_S3hats_m0p2000 = np.sqrt(err_sq)

In [ ]:
mass="0p3000"

err_sq = 0.0*windows2[mass]
for jdrop in np.arange(nbetas[mass]):
    S3hats_jkresamp = S3hats_jkdatas_m0p3000[jdrop]
    
    jk = Jackknife( nbins, 1 )
    jk.set( jk_avg, S3hats_jkresamp )
    
    jk.do_it()
    # jk.do_it_wavg( S3hats_m0p3000 )
    err_sq += jk.err()**2

delta_S3hats_m0p3000 = np.sqrt(err_sq)

In [ ]:
mass="0p4000"

chk=[]

err_sq = 0.0*windows2[mass]
for jdrop in np.arange(nbetas[mass]):
    S3hats_jkresamp = S3hats_jkdatas_m0p4000[jdrop]
    
    jk = Jackknife( nbins, 1 )
    jk.set( jk_avg, S3hats_jkresamp )
    
    jk.do_it()
    # jk.do_it_wavg( S3hats_m0p4000 )
    err_sq += jk.err()**2
    chk.append(jk.mean())

delta_S3hats_m0p4000 = np.sqrt(err_sq)

In [ ]:
# plt.clf()
# for i in range(9):
#     plt.plot( windows2[mass], np.array(chk)[i], label=str(i), ls='none', marker='o' )

# plt.legend()
# plt.show()

In [ ]:
# plt.clf()
# for i in range(5,9):
#     plt.plot( windows2[mass], np.array(chk)[i], label=str(i) )

# plt.legend()
# plt.show()

In [ ]:
# plt.clf()
# for i in range(3,6):
#     plt.plot( windows2[mass], np.array(chk)[i], label=str(i) )

# plt.legend()
# plt.show()

In [ ]:
tmp=S3hats_jkdatas_m0p2000[2]
plt.clf()
for ibin in range(0,16):
    plt.plot( tmp[ibin], label=str(ibin) )

plt.legend()
plt.show()

In [ ]:
windows2[mass].shape

In [ ]:
plt.clf()
mm=29
for i in range(4,nbetas[mass]):
    plt.plot( windows2[mass][mm:], np.array(chk)[i][mm:], label=str(i) )

plt.legend()
plt.show()

In [ ]:
# basedir = "../spline_cpp_v16_32c_claude/"

In [ ]:
# nbeta=nbetas[mass]

In [ ]:
# betac_ibetac=np.loadtxt(basedir+"/betac_ibetac_mass"+mass+"_"+str(Ns)+".dat")
# ibetac_jk=np.loadtxt(basedir+"/ibetac_jk_mass"+mass+"_"+str(Ns)+".dat")

In [ ]:
# tmp=[]

# for jdrop in range(nbeta):
#     for ibin in range(nbins):
#         i=int(ibetac_jk[jdrop][ibin])
#         minima_data_=np.loadtxt(basedir+"/fit_params_"+str(Ns)+"c_m"+mass+"_jk_"+str(jdrop)+"_40_"+str(ibin)+"/minima_data_"+str(i)+".dat")
#         tmp.append(minima_data_)

In [ ]:
# plt.clf()
# plt.plot( np.array(tmp).T[1])
# plt.legend()
# plt.show()

In [ ]:
fitparams

In [ ]:
def S3hat_c2only( TmTc_MB, Mc, c2 ):
    return S3hat_fitter( TmTc_MB, Mc, 0.0, 0.0, c2, GAM )   # pure thin-wall: c0=c1=0

fitparams0 = [2.13, 0.0, 0.0, 3.0e-06, GAM]   # c2-only: fit [Mc,c2]; slots c0,c1 held at 0


In [ ]:
# k1=20
# k2=18 # 15
# k3=20

k1=20
k2=18 # 15
k3=26

In [ ]:
mass='0p2000'
betas_all = np.loadtxt( "../spline_cpp_v16_32c_claude/betas_mass"+mass+"_32.dat" )
betas_all[windows2[mass]][-k1:]

In [ ]:
mass='0p3000'
betas_all = np.loadtxt( "../spline_cpp_v16_32c_claude/betas_mass"+mass+"_32.dat" )
betas_all[windows2[mass]][-k2:]

In [ ]:
mass='0p4000'
betas_all = np.loadtxt( "../spline_cpp_v16_32c_claude/betas_mass"+mass+"_32.dat" )
betas_all[windows2[mass]][-k3:]

In [ ]:
S3hats_m0p4000.shape

In [ ]:
xs = np.array([])
delta_xs = np.array([])
ys = np.array([])
delta_ys = np.array([])
zs = np.array([])
delta_zs = np.array([])

ks=[k1,k2,k3]
zpool = [ S3hats_m0p2000, S3hats_m0p3000, S3hats_m0p4000]
delta_zpool = [ delta_S3hats_m0p2000, delta_S3hats_m0p3000, delta_S3hats_m0p4000]

counter=0
for row in x_dx_y_dy:
    k=ks[counter]
    xs = np.concatenate( [ xs, row[0][-k:] ] )
    delta_xs = np.concatenate( [ delta_xs, row[1][-k:] ] )
    ys = np.concatenate( [ ys, row[2][-k:] ] )
    delta_ys = np.concatenate( [ delta_ys, row[3][-k:] ] )
    zs = np.concatenate( [ zs, zpool[counter][-k:] ] )
    delta_zs = np.concatenate( [ delta_zs, delta_zpool[counter][-k:] ] )
    counter+=1

fitparams = fitparams0
for i in range(3):
    d0s = d0_S3hat_fitter( [xs,ys], fitparams[0], fitparams[1], fitparams[2], fitparams[3], fitparams[4] )
    d1s = d1_S3hat_fitter( [xs,ys], fitparams[0], fitparams[1], fitparams[2], fitparams[3], fitparams[4] )
    sigma = np.sqrt( delta_zs**2 + d0s**2 * delta_xs**2 + d1s**2 * delta_ys**2 )
    opt = sp.optimize.curve_fit( f=S3hat_c2only, xdata=[xs,ys], ydata=zs, 
                                 sigma=sigma,
                                  p0=[fitparams[0], fitparams[3]],
                               full_output=True, 
                               absolute_sigma=True)
    fitparams = [opt[0][0], 0.0, 0.0, opt[0][1], GAM]        
S3params=fitparams

In [ ]:
sigma

In [ ]:
yyyy = S3hat_fitter( [xs,ys], fitparams[0], fitparams[1], fitparams[2], fitparams[3], fitparams[4] )

In [ ]:
np.sum( (yyyy-zs)**2/sigma**2 )/( len(yyyy)-5 )*epsilon**2

In [ ]:
fig = plt.figure()
ax = plt.axes(projection='3d')

ax.errorbar( xs, ys, zs,
           xerr=delta_xs/epsilon,
           yerr=delta_ys/epsilon,
           zerr=delta_zs/epsilon, ls='none', c='orange', marker='o'
           , zorder=-32, alpha=0.6, capsize=4
           )

# ax.scatter( x1, x2, yy )
# zuplims=yy+dyy
# zlolims=yy-dyy
# estep = 1
# ax.errorbar( x1, x2, yy, 0.01, zuplims=zuplims, zlolims=zlolims, errorevery=estep, ls='none')
# ax.scatter( x1, x2, fit_yy )

xx1 = np.repeat(np.linspace(-0.0006, -0.00002, 200), 200).reshape(200,200)
xx2 = np.repeat(np.linspace(2.3, 3.9, 200), 200).reshape(200,200).T # 2.35, ...
zs2 = S3hat_fitter( [xx1, xx2], fitparams[0], fitparams[1], fitparams[2], fitparams[3], fitparams[4] )
ax.scatter( xx1, xx2, zs2, alpha=0.05, marker='.' )

ax.plot( xs, ys, zs, ls='none', c='orange', marker='o' )

ax.view_init(elev=25, azim=-80, roll=0)

ax.set_zlim(0,100)
ax.set_xlabel("$T-T_c$")
ax.set_ylabel("$M_B$")
ax.set_zlabel("$S^{\\prime}_3$")

# plt.savefig("S3_"+gamtag+".pdf", bbox_inches='tight')

plt.show()

In [ ]:
np.savetxt( "S3_MC_"+gamtag+".dat", np.array([xs, ys, zs, delta_xs/epsilon, delta_ys/epsilon, delta_zs/epsilon]).T )

In [ ]:
delta_xs

In [ ]:
# vary m=0.2
jk_S3params_m0p2000=[]
mass="0p2000"

for jdrop in np.arange(nbetas[mass]):
    fp_jk_=[]
    # S3hats_jkresamp = eps_trick(S3hats_jkdatas_m0p2000[jdrop], eps)
    S3hats_jkresamp = S3hats_jkdatas_m0p2000[jdrop]
    for ibin in np.arange(nbins):
        xs = np.array([])
        delta_xs = np.array([])
        ys = np.array([])
        delta_ys = np.array([])
        zs = np.array([])
        delta_zs = np.array([])

        ks=[k1,k2,k3]
        zpool = [ S3hats_jkresamp[ibin], S3hats_m0p3000, S3hats_m0p4000]
        delta_zpool = [ delta_S3hats_m0p2000, delta_S3hats_m0p3000, delta_S3hats_m0p4000]
        
        counter=0
        for row in x_dx_y_dy:
            k=ks[counter]
            xs = np.concatenate( [ xs, row[0][-k:] ] )
            delta_xs = np.concatenate( [ delta_xs, row[1][-k:] ] )
            ys = np.concatenate( [ ys, row[2][-k:] ] )
            delta_ys = np.concatenate( [ delta_ys, row[3][-k:] ] )
            zs = np.concatenate( [ zs, zpool[counter][-k:] ] )
            delta_zs = np.concatenate( [ delta_zs, delta_zpool[counter][-k:] ] )
            counter+=1
        
        fitparams = fitparams0
        for i in range(3):
            d0s = d0_S3hat_fitter( [xs,ys], fitparams[0], fitparams[1], fitparams[2], fitparams[3], fitparams[4] )
            d1s = d1_S3hat_fitter( [xs,ys], fitparams[0], fitparams[1], fitparams[2], fitparams[3], fitparams[4] )
            sigma = np.sqrt( delta_zs**2 + d0s**2 * delta_xs**2 + d1s**2 * delta_ys**2 )
            opt = sp.optimize.curve_fit( f=S3hat_c2only, xdata=[xs,ys], ydata=zs, 
                                         sigma=sigma,
                                          p0=[fitparams[0], fitparams[3]],
                                       full_output=True, 
                                       absolute_sigma=True)
            fitparams = [opt[0][0], 0.0, 0.0, opt[0][1], GAM]        
        fp_jk_.append( fitparams )
    
    jk_data = np.array(fp_jk_)
    jk_S3params_m0p2000.append( jk_data )

In [ ]:
# vary m=0.3
jk_S3params_m0p3000=[]
mass="0p3000"

for jdrop in np.arange(nbetas[mass]):
    fp_jk_=[]
    # S3hats_jkresamp = eps_trick(S3hats_jkdatas_m0p3000[jdrop], eps)
    S3hats_jkresamp = S3hats_jkdatas_m0p3000[jdrop]
    for ibin in np.arange(nbins):
        xs = np.array([])
        delta_xs = np.array([])
        ys = np.array([])
        delta_ys = np.array([])
        zs = np.array([])
        delta_zs = np.array([])

        ks=[k1,k2,k3]
        zpool = [ S3hats_m0p2000, S3hats_jkresamp[ibin], S3hats_m0p4000]
        delta_zpool = [ delta_S3hats_m0p2000, delta_S3hats_m0p3000, delta_S3hats_m0p4000]
        
        counter=0
        for row in x_dx_y_dy:
            k=ks[counter]
            xs = np.concatenate( [ xs, row[0][-k:] ] )
            delta_xs = np.concatenate( [ delta_xs, row[1][-k:] ] )
            ys = np.concatenate( [ ys, row[2][-k:] ] )
            delta_ys = np.concatenate( [ delta_ys, row[3][-k:] ] )
            zs = np.concatenate( [ zs, zpool[counter][-k:] ] )
            delta_zs = np.concatenate( [ delta_zs, delta_zpool[counter][-k:] ] )
            counter+=1
        
        fitparams = fitparams0
        for i in range(3):
            d0s = d0_S3hat_fitter( [xs,ys], fitparams[0], fitparams[1], fitparams[2], fitparams[3], fitparams[4] )
            d1s = d1_S3hat_fitter( [xs,ys], fitparams[0], fitparams[1], fitparams[2], fitparams[3], fitparams[4] )
            sigma = np.sqrt( delta_zs**2 + d0s**2 * delta_xs**2 + d1s**2 * delta_ys**2 )
            opt = sp.optimize.curve_fit( f=S3hat_c2only, xdata=[xs,ys], ydata=zs, 
                                         sigma=sigma,
                                          p0=[fitparams[0], fitparams[3]],
                                       full_output=True, 
                                       absolute_sigma=True)
            fitparams = [opt[0][0], 0.0, 0.0, opt[0][1], GAM]        
        fp_jk_.append( fitparams )
    
    jk_data = np.array(fp_jk_)
    jk_S3params_m0p3000.append( jk_data )

In [ ]:
# vary m=0.4
jk_S3params_m0p4000=[]
mass="0p4000"

for jdrop in np.arange(nbetas[mass]):
    fp_jk_=[]
    # S3hats_jkresamp = eps_trick(S3hats_jkdatas_m0p4000[jdrop], eps)
    S3hats_jkresamp = S3hats_jkdatas_m0p4000[jdrop]
    for ibin in np.arange(nbins):
        xs = np.array([])
        delta_xs = np.array([])
        ys = np.array([])
        delta_ys = np.array([])
        zs = np.array([])
        delta_zs = np.array([])

        ks=[k1,k2,k3]
        zpool = [ S3hats_m0p2000, S3hats_m0p3000, S3hats_jkresamp[ibin] ]
        delta_zpool = [ delta_S3hats_m0p2000, delta_S3hats_m0p3000, delta_S3hats_m0p4000]
        
        counter=0
        for row in x_dx_y_dy:
            k=ks[counter]
            xs = np.concatenate( [ xs, row[0][-k:] ] )
            delta_xs = np.concatenate( [ delta_xs, row[1][-k:] ] )
            ys = np.concatenate( [ ys, row[2][-k:] ] )
            delta_ys = np.concatenate( [ delta_ys, row[3][-k:] ] )
            zs = np.concatenate( [ zs, zpool[counter][-k:] ] )
            delta_zs = np.concatenate( [ delta_zs, delta_zpool[counter][-k:] ] )
            counter+=1
        
        fitparams = fitparams0
        for i in range(3):
            d0s = d0_S3hat_fitter( [xs,ys], fitparams[0], fitparams[1], fitparams[2], fitparams[3], fitparams[4] )
            d1s = d1_S3hat_fitter( [xs,ys], fitparams[0], fitparams[1], fitparams[2], fitparams[3], fitparams[4] )
            sigma = np.sqrt( delta_zs**2 + d0s**2 * delta_xs**2 + d1s**2 * delta_ys**2 )
            opt = sp.optimize.curve_fit( f=S3hat_c2only, xdata=[xs,ys], ydata=zs, 
                                         sigma=sigma,
                                          p0=[fitparams[0], fitparams[3]],
                                       full_output=True, 
                                       absolute_sigma=True)
            fitparams = [opt[0][0], 0.0, 0.0, opt[0][1], GAM]        
        fp_jk_.append( fitparams )
    
    jk_data = np.array(fp_jk_)
    jk_S3params_m0p4000.append( jk_data )

In [ ]:
class F_S3hat:
    def fitter(self, TmTc_MB, Mc, c0, c1, c2, gam ):
        TmTc = TmTc_MB[0]
        MB = TmTc_MB[1]
        return (MB-Mc)**gam  * ( c0 + c1/TmTc + c2/TmTc**2 )

    def __init__( self, fp ):
        self.fp = fp

    def __call__( self, TmTc, MB ):
        return self.fitter( [TmTc, MB], self.fp[0], self.fp[1], self.fp[2], self.fp[3], self.fp[4] )

    def dT( self, TmTc, MB ):
        Mc = self.fp[0]
        c0 = self.fp[1]
        c1 = self.fp[2]
        c2 = self.fp[3]
        gam = self.fp[4]
        return (MB-Mc)**gam  * ( -1.0*c1/TmTc**2 -2.0 * c2/TmTc**3 )

# DV

In [ ]:
if two_param:
    def DeltaVhat_fitter(TmTc_MB, a0, b0):
        TmTc = TmTc_MB[0]
        MB = TmTc_MB[1]
        return TmTc * ( a0 + b0*MB ) + TmTc**2 * ( a0 + b0*MB )
else:
    def DeltaVhat_fitter(TmTc_MB, a0, b0, c0, d0):
        TmTc = TmTc_MB[0]
        MB = TmTc_MB[1]
        return TmTc * ( a0 + b0*MB ) + TmTc**2 * ( c0 + d0*MB )

    def d0_DeltaVhat_fitter(TmTc_MB, a0, b0, c0, d0):
        TmTc = TmTc_MB[0]
        MB = TmTc_MB[1]
        return ( a0 + b0*MB ) + 2.0*TmTc * ( c0 + d0*MB )

    def d1_DeltaVhat_fitter(TmTc_MB, a0, b0, c0, d0):
        TmTc = TmTc_MB[0]
        MB = TmTc_MB[1]
        return TmTc * ( b0 ) + TmTc**2 * ( d0 )

In [ ]:
pwd

In [ ]:
mass="0p2000"

minima = []
directory = "../spline_cpp_v16_32c_claude/fit_params_"+str(Ns)+"c_m"+mass+"/"
f = h5py.File(directory+"/Veff.h5", 'r')
for ibeta in np.arange(nbetasMeas):
    # desc = "jk_"+str(jdrop)+"_40_"+str(ibin)
    tmp = f[str(ibeta)+'/minima_data'][()]
    minima.append(tmp)
f.close()

DeltaVhats_ = []
for ibeta in windows[mass]:
    DeltaVhat = minima[ibeta][3] - minima[ibeta][2]
    DeltaVhats_.append( DeltaVhat )
DeltaVhats_m0p2000 = np.array(DeltaVhats_)

# get error for each ibeta (renormalize) from jackknife
ibetacs_jk = np.loadtxt( "../spline_cpp_v16_32c_claude/ibetac_jk_mass"+mass+"_"+str(Ns)+".dat" ) # renorm Tc
 
# err_sq = 0.0*windows2[mass]
DeltaVhats_jkdatas_m0p2000= []

for jdrop in np.arange(nbetas[mass]):

    jk_data_ = []
    for ibin in np.arange(nbins):
        minima = []
        desc = "jk_"+str(jdrop)+"_40_"+str(ibin)
        directory = "../spline_cpp_v16_32c_claude/fit_params_"+str(Ns)+"c_m"+mass+"_"+desc+"/"
        f = h5py.File(directory+"/Veff.h5", 'r')
        for ibeta in np.arange(nbetasMeas):
            tmp = f[str(ibeta)+'/minima_data'][()]
            
            # tmp = np.loadtxt("../spline_cpp_v16_32c_claude/fit_params_32c_m"+mass+"_jk_"+str(jdrop)+"_40_"+str(ibin)+"/minima_data_"+str(ibeta)+".dat")
            minima.append(tmp)
        f.close()
    
        dibeta = int(ibetacs_jk[jdrop][ibin]) - ibetac0s[mass]
    
        DeltaVhats_ = []
        for ibeta in windows[mass] + dibeta:
            DeltaVhat = minima[ibeta][3] - minima[ibeta][2]
            DeltaVhats_.append( DeltaVhat )
        DeltaVhats = np.array(DeltaVhats_)
        jk_data_.append( DeltaVhats )
    jk_data = np.array( jk_data_ )

    DeltaVhats_jkdatas_m0p2000.append( jk_data )

In [ ]:
mass="0p3000"

minima = []
directory = "../spline_cpp_v16_32c_claude/fit_params_"+str(Ns)+"c_m"+mass+"/"
f = h5py.File(directory+"/Veff.h5", 'r')
for ibeta in np.arange(nbetasMeas):
    tmp = f[str(ibeta)+'/minima_data'][()]
    minima.append(tmp)
f.close()

DeltaVhats_ = []
for ibeta in windows[mass]:
    DeltaVhat = minima[ibeta][3] - minima[ibeta][2]
    DeltaVhats_.append( DeltaVhat )
DeltaVhats_m0p3000 = np.array(DeltaVhats_)

# get error for each ibeta (renormalize) from jackknife
ibetacs_jk = np.loadtxt( "../spline_cpp_v16_32c_claude/ibetac_jk_mass"+mass+"_"+str(Ns)+".dat" ) # renorm Tc

# err_sq = 0.0*windows2[mass]
DeltaVhats_jkdatas_m0p3000= []

for jdrop in np.arange(nbetas[mass]):

    jk_data_ = []
    for ibin in np.arange(nbins):
        minima = []
        desc = "jk_"+str(jdrop)+"_40_"+str(ibin)
        directory = "../spline_cpp_v16_32c_claude/fit_params_"+str(Ns)+"c_m"+mass+"_"+desc+"/"
        f = h5py.File(directory+"/Veff.h5", 'r')
        for ibeta in np.arange(nbetasMeas):
            tmp = f[str(ibeta)+'/minima_data'][()]
            minima.append(tmp)
        f.close()
    
        dibeta = int(ibetacs_jk[jdrop][ibin]) - ibetac0s[mass]
    
        DeltaVhats_ = []
        for ibeta in windows[mass] + dibeta:
            DeltaVhat = minima[ibeta][3] - minima[ibeta][2]
            DeltaVhats_.append( DeltaVhat )
        DeltaVhats = np.array(DeltaVhats_)
        jk_data_.append( DeltaVhats )
    jk_data = np.array( jk_data_ )

    DeltaVhats_jkdatas_m0p3000.append( jk_data )

In [ ]:
mass="0p4000"

minima = []
directory = "../spline_cpp_v16_32c_claude/fit_params_"+str(Ns)+"c_m"+mass+"/"
f = h5py.File(directory+"/Veff.h5", 'r')
for ibeta in np.arange(nbetasMeas):
    tmp = f[str(ibeta)+'/minima_data'][()]
    minima.append(tmp)
f.close()

DeltaVhats_ = []
for ibeta in windows[mass]:
    DeltaVhat = minima[ibeta][3] - minima[ibeta][2]
    DeltaVhats_.append( DeltaVhat )
DeltaVhats_m0p4000 = np.array(DeltaVhats_)

# get error for each ibeta (renormalize) from jackknife
ibetacs_jk = np.loadtxt( "../spline_cpp_v16_32c_claude/ibetac_jk_mass"+mass+"_"+str(Ns)+".dat" ) # renorm Tc

# err_sq = 0.0*windows2[mass]
DeltaVhats_jkdatas_m0p4000= []

for jdrop in np.arange(nbetas[mass]):

    jk_data_ = []
    for ibin in np.arange(nbins):
        # print( jdrop, ibin )
        minima = []
        desc = "jk_"+str(jdrop)+"_40_"+str(ibin)
        directory = "../spline_cpp_v16_32c_claude/fit_params_"+str(Ns)+"c_m"+mass+"_"+desc+"/"
        f = h5py.File(directory+"/Veff.h5", 'r')
        for ibeta in np.arange(nbetasMeas):
            tmp = f[str(ibeta)+'/minima_data'][()]
            minima.append(tmp)
        f.close()
    
        dibeta = int(ibetacs_jk[jdrop][ibin]) - ibetac0s[mass]
    
        DeltaVhats_ = []
        for ibeta in windows[mass] + dibeta:
            DeltaVhat = minima[ibeta][3] - minima[ibeta][2]
            DeltaVhats_.append( DeltaVhat )
        DeltaVhats = np.array(DeltaVhats_)
        jk_data_.append( DeltaVhats )
    jk_data = np.array( jk_data_ )

    DeltaVhats_jkdatas_m0p4000.append( jk_data )

In [ ]:
x_dx_y_dy = []

# 0p2
mq = 0.2
mass="0p2000"

betas_all = np.loadtxt( "../spline_cpp_v16_32c_claude/betas_mass"+mass+"_32.dat" )
betac = betas_all[ibetac0s[mass]]
betas = betas_all[windows[mass]]

MBs = f_MB( betas, mq )
DMBs = np.array([f_MB.D( beta, mq ) for beta in betas ])

dbetas = betas - betac
TmTcs = [f_TmTc( dbeta, mq ) for dbeta in dbetas]
DTmTcs = [f_TmTc.D( dbeta, mq ) for dbeta in dbetas]

x_dx_y_dy.append( [ TmTcs, DTmTcs, MBs, DMBs ] )


# 0p3
mq = 0.3
mass="0p3000"

betas_all = np.loadtxt( "../spline_cpp_v16_32c_claude/betas_mass"+mass+"_32.dat" )
betac = betas_all[ibetac0s[mass]]
betas = betas_all[windows[mass]]

MBs = f_MB( betas, mq )
DMBs = np.array([f_MB.D( beta, mq ) for beta in betas ])

dbetas = betas - betac
TmTcs = [f_TmTc( dbeta, mq ) for dbeta in dbetas]
DTmTcs = [f_TmTc.D( dbeta, mq ) for dbeta in dbetas]

x_dx_y_dy.append( [ TmTcs, DTmTcs, MBs, DMBs ] )


# 0p4
mq = 0.4
mass="0p4000"

betas_all = np.loadtxt( "../spline_cpp_v16_32c_claude/betas_mass"+mass+"_32.dat" )
betac = betas_all[ibetac0s[mass]]
betas = betas_all[windows[mass]]

MBs = f_MB( betas, mq )
DMBs = np.array([f_MB.D( beta, mq ) for beta in betas ])

dbetas = betas - betac
TmTcs = [f_TmTc( dbeta, mq ) for dbeta in dbetas]
DTmTcs = [f_TmTc.D( dbeta, mq ) for dbeta in dbetas]

x_dx_y_dy.append( [ TmTcs, DTmTcs, MBs, DMBs ] )

In [ ]:
len(x_dx_y_dy)

In [ ]:
mass="0p2000"

err_sq = 0.0*windows[mass]
for jdrop in np.arange(nbetas[mass]):
    DeltaVhats_jkresamp = DeltaVhats_jkdatas_m0p2000[jdrop]
    
    jk = Jackknife( nbins, 1 )
    jk.set( jk_avg, DeltaVhats_jkresamp )
    
    jk.do_it()
    # jk.do_it_wavg( DeltaVhats_m0p2000 )
    err_sq += jk.err()**2
    
delta_DeltaVhats_m0p2000 = np.sqrt(err_sq)

In [ ]:
mass="0p3000"

err_sq = 0.0*windows[mass]
for jdrop in np.arange(nbetas[mass]):
    DeltaVhats_jkresamp = DeltaVhats_jkdatas_m0p3000[jdrop]
    
    jk = Jackknife( nbins, 1 )
    jk.set( jk_avg, DeltaVhats_jkresamp )
    
    jk.do_it()
    # jk.do_it_wavg( DeltaVhats_m0p3000 )
    err_sq += jk.err()**2

delta_DeltaVhats_m0p3000 = np.sqrt(err_sq)

In [ ]:
mass="0p4000"

err_sq = 0.0*windows[mass]
for jdrop in np.arange(nbetas[mass]):
    DeltaVhats_jkresamp = DeltaVhats_jkdatas_m0p4000[jdrop]
    
    jk = Jackknife( nbins, 1 )
    jk.set( jk_avg, DeltaVhats_jkresamp )
    
    jk.do_it()
    # jk.do_it_wavg( DeltaVhats_m0p4000 )
    err_sq += jk.err()**2

delta_DeltaVhats_m0p4000 = np.sqrt(err_sq)

In [ ]:
if two_param:
    fitparams0=[ 13.7027236, -9.83191225 ]
    k1=40
    k2=25
    k3=30
else:
    fitparams0=[2.23845161e+01, -1.34122960e+01,  3.46978544e+04, -1.26978924e+04]
    k1=40
    k2=38
    k3=44

In [ ]:
mass='0p2000'
betas_all = np.loadtxt( "../spline_cpp_v16_32c_claude/betas_mass"+mass+"_32.dat" )
betas_all[windows[mass]][-k1:]

In [ ]:
mass='0p3000'
betas_all = np.loadtxt( "../spline_cpp_v16_32c_claude/betas_mass"+mass+"_32.dat" )
betas_all[windows[mass]][-k2:]

In [ ]:
mass='0p4000'
betas_all = np.loadtxt( "../spline_cpp_v16_32c_claude/betas_mass"+mass+"_32.dat" )
betas_all[windows[mass]][-k3:]

In [ ]:
xs = np.array([])
delta_xs = np.array([])
ys = np.array([])
delta_ys = np.array([])
zs = np.array([])
delta_zs = np.array([])

ks=[k1,k2,k3]
zpool = [ DeltaVhats_m0p2000, DeltaVhats_m0p3000, DeltaVhats_m0p4000]
delta_zpool = [ delta_DeltaVhats_m0p2000, delta_DeltaVhats_m0p3000, delta_DeltaVhats_m0p4000]

counter=0
for row in x_dx_y_dy:
    k=ks[counter]
    xs = np.concatenate( [ xs, row[0][-k:] ] )
    delta_xs = np.concatenate( [ delta_xs, row[1][-k:] ] )
    ys = np.concatenate( [ ys, row[2][-k:] ] )
    delta_ys = np.concatenate( [ delta_ys, row[3][-k:] ] )
    zs = np.concatenate( [ zs, zpool[counter][-k:] ] )
    delta_zs = np.concatenate( [ delta_zs, delta_zpool[counter][-k:] ] )
    counter+=1

fitparams = fitparams0
for i in range(3):
    d0s = d0_DeltaVhat_fitter( [xs,ys], fitparams[0], fitparams[1], fitparams[2], fitparams[3] )
    d1s = d1_DeltaVhat_fitter( [xs,ys], fitparams[0], fitparams[1], fitparams[2], fitparams[3] )
    sigma = np.sqrt( delta_zs**2 + d0s**2 * delta_xs**2 + d1s**2 * delta_ys**2 )
    opt = sp.optimize.curve_fit( f=DeltaVhat_fitter, xdata=[xs,ys], ydata=zs, 
                                 sigma=sigma,
                          p0=fitparams,
                               full_output=True,
                               absolute_sigma=True)
    fitparams = opt[0]        
DeltaVparams=opt[0]

In [ ]:
yyyy = DeltaVhat_fitter( [xs,ys], fitparams[0], fitparams[1], fitparams[2], fitparams[3] )

In [ ]:
np.sum( (yyyy-zs)**2/sigma**2 )/( len(yyyy)-5 )*epsilon**2

In [ ]:
pwd

In [ ]:
np.savetxt( "DeltaV_MC_"+gamtag+".dat", np.array([xs, ys, zs, delta_xs/epsilon, delta_ys/epsilon, delta_zs/epsilon]).T )

In [ ]:
fitparams

In [ ]:
fig = plt.figure()
ax = plt.axes(projection='3d')

ax.errorbar( xs, ys, zs,
           xerr=delta_xs/epsilon,
           yerr=delta_ys/epsilon,
           zerr=delta_zs/epsilon, ls='none', c='orange', marker='d', 
             zorder=0, alpha=0.6, capsize=4
           )


xx1 = np.repeat(np.linspace(-0.0006, -0.00002, 100), 100).reshape(100,100)
xx2 = np.repeat(np.linspace(2.3, 3.9, 100), 100).reshape(100,100).T
zs2 = DeltaVhat_fitter( [xx1, xx2], fitparams[0], fitparams[1], fitparams[2], fitparams[3] )
ax.scatter( xx1, xx2, zs2, alpha=0.1, marker='.'
             )

ax.scatter( xs, ys, zs )

ax.view_init(elev=25, azim=-40, roll=0)

# ax.set_zlim(0,100)
ax.set_xlabel("$T-T_c$")
ax.set_ylabel("$M_B$")
ax.set_zlabel("$\\Delta V'$")

plt.savefig("DeltaV_"+gamtag+".pdf")

plt.show()

In [ ]:
# vary m=0.2
jk_DeltaVparams_m0p2000=[]
mass="0p2000"

for jdrop in np.arange(nbetas[mass]):
    fp_jk_=[]
    # DeltaVhats_jkresamp = eps_trick(DeltaVhats_jkdatas_m0p2000[jdrop], eps)
    DeltaVhats_jkresamp = DeltaVhats_jkdatas_m0p2000[jdrop]
    for ibin in np.arange(nbins):
        xs = np.array([])
        delta_xs = np.array([])
        ys = np.array([])
        delta_ys = np.array([])
        zs = np.array([])
        delta_zs = np.array([])

        ks=[k1,k2,k3]
        zpool = [ DeltaVhats_jkresamp[ibin], DeltaVhats_m0p3000, DeltaVhats_m0p4000]
        delta_zpool = [ delta_DeltaVhats_m0p2000, delta_DeltaVhats_m0p3000, delta_DeltaVhats_m0p4000]
        
        counter=0
        for row in x_dx_y_dy:
            k=ks[counter]
            xs = np.concatenate( [ xs, row[0][-k:] ] )
            delta_xs = np.concatenate( [ delta_xs, row[1][-k:] ] )
            ys = np.concatenate( [ ys, row[2][-k:] ] )
            delta_ys = np.concatenate( [ delta_ys, row[3][-k:] ] )
            zs = np.concatenate( [ zs, zpool[counter][-k:] ] )
            delta_zs = np.concatenate( [ delta_zs, delta_zpool[counter][-k:] ] )
            counter+=1
        
        fitparams = fitparams0
        for i in range(3):
            d0s = d0_DeltaVhat_fitter( [xs,ys], fitparams[0], fitparams[1], fitparams[2], fitparams[3] )
            d1s = d1_DeltaVhat_fitter( [xs,ys], fitparams[0], fitparams[1], fitparams[2], fitparams[3] )
            sigma = np.sqrt( delta_zs**2 + d0s**2 * delta_xs**2 + d1s**2 * delta_ys**2 )
            opt = sp.optimize.curve_fit( f=DeltaVhat_fitter, xdata=[xs,ys], ydata=zs, 
                                         sigma=sigma,
                                  p0=fitparams,
                                       full_output=True,
                                       absolute_sigma=True)
            fitparams = opt[0]        
        fp_jk_.append( opt[0] )
    
    jk_data = np.array(fp_jk_)
    jk_DeltaVparams_m0p2000.append( jk_data )

In [ ]:
# vary m=0.3
jk_DeltaVparams_m0p3000=[]
mass="0p3000"

for jdrop in np.arange(nbetas[mass]):
    fp_jk_=[]
    # DeltaVhats_jkresamp = eps_trick(DeltaVhats_jkdatas_m0p3000[jdrop], eps)
    DeltaVhats_jkresamp = DeltaVhats_jkdatas_m0p3000[jdrop]
    for ibin in np.arange(nbins):
        xs = np.array([])
        delta_xs = np.array([])
        ys = np.array([])
        delta_ys = np.array([])
        zs = np.array([])
        delta_zs = np.array([])

        ks=[k1,k2,k3]
        zpool = [ DeltaVhats_m0p2000, DeltaVhats_jkresamp[ibin], DeltaVhats_m0p4000]
        delta_zpool = [ delta_DeltaVhats_m0p2000, delta_DeltaVhats_m0p3000, delta_DeltaVhats_m0p4000]
        
        counter=0
        for row in x_dx_y_dy:
            k=ks[counter]
            xs = np.concatenate( [ xs, row[0][-k:] ] )
            delta_xs = np.concatenate( [ delta_xs, row[1][-k:] ] )
            ys = np.concatenate( [ ys, row[2][-k:] ] )
            delta_ys = np.concatenate( [ delta_ys, row[3][-k:] ] )
            zs = np.concatenate( [ zs, zpool[counter][-k:] ] )
            delta_zs = np.concatenate( [ delta_zs, delta_zpool[counter][-k:] ] )
            counter+=1
        
        fitparams = fitparams0
        for i in range(3):
            d0s = d0_DeltaVhat_fitter( [xs,ys], fitparams[0], fitparams[1], fitparams[2], fitparams[3] )
            d1s = d1_DeltaVhat_fitter( [xs,ys], fitparams[0], fitparams[1], fitparams[2], fitparams[3] )
            sigma = np.sqrt( delta_zs**2 + d0s**2 * delta_xs**2 + d1s**2 * delta_ys**2 )
            opt = sp.optimize.curve_fit( f=DeltaVhat_fitter, xdata=[xs,ys], ydata=zs, 
                                         sigma=sigma,
                                  p0=fitparams,
                                       full_output=True,
                                       absolute_sigma=True)
            fitparams = opt[0]        
        fp_jk_.append( opt[0] )
    
    jk_data = np.array(fp_jk_)
    jk_DeltaVparams_m0p3000.append( jk_data )

In [ ]:
# vary m=0.4
jk_DeltaVparams_m0p4000=[]
mass="0p4000"

for jdrop in np.arange(nbetas[mass]):
    fp_jk_=[]
    # DeltaVhats_jkresamp = eps_trick(DeltaVhats_jkdatas_m0p4000[jdrop], eps)
    DeltaVhats_jkresamp = DeltaVhats_jkdatas_m0p4000[jdrop]
    for ibin in np.arange(nbins):
        xs = np.array([])
        delta_xs = np.array([])
        ys = np.array([])
        delta_ys = np.array([])
        zs = np.array([])
        delta_zs = np.array([])

        ks=[k1,k2,k3]
        zpool = [ DeltaVhats_m0p2000, DeltaVhats_m0p3000, DeltaVhats_jkresamp[ibin] ]
        delta_zpool = [ delta_DeltaVhats_m0p2000, delta_DeltaVhats_m0p3000, delta_DeltaVhats_m0p4000]
        
        counter=0
        for row in x_dx_y_dy:
            k=ks[counter]
            xs = np.concatenate( [ xs, row[0][-k:] ] )
            delta_xs = np.concatenate( [ delta_xs, row[1][-k:] ] )
            ys = np.concatenate( [ ys, row[2][-k:] ] )
            delta_ys = np.concatenate( [ delta_ys, row[3][-k:] ] )
            zs = np.concatenate( [ zs, zpool[counter][-k:] ] )
            delta_zs = np.concatenate( [ delta_zs, delta_zpool[counter][-k:] ] )
            counter+=1
        
        fitparams = fitparams0
        for i in range(3):
            d0s = d0_DeltaVhat_fitter( [xs,ys], fitparams[0], fitparams[1], fitparams[2], fitparams[3] )
            d1s = d1_DeltaVhat_fitter( [xs,ys], fitparams[0], fitparams[1], fitparams[2], fitparams[3] )
            sigma = np.sqrt( delta_zs**2 + d0s**2 * delta_xs**2 + d1s**2 * delta_ys**2 )
            opt = sp.optimize.curve_fit( f=DeltaVhat_fitter, xdata=[xs,ys], ydata=zs, 
                                         sigma=sigma,
                                  p0=fitparams,
                                       full_output=True,
                                       absolute_sigma=True)
            fitparams = opt[0]        
        fp_jk_.append( opt[0] )
    
    jk_data = np.array(fp_jk_)
    jk_DeltaVparams_m0p4000.append( jk_data )

In [ ]:
# @@@@@@@@@@@@@@

# GW

In [ ]:
h = h5py.File('Tcparams_'+gamtag+'.hdf5', 'w')
dset = h.create_dataset('mean', data=np.array(Tcparams))
dset = h.create_dataset('jk_m0p2000', data=np.array(jk_Tcparams_m0p2000))
dset = h.create_dataset('jk_m0p3000', data=np.array(jk_Tcparams_m0p3000))
dset = h.create_dataset('jk_m0p4000', data=np.array(jk_Tcparams_m0p4000))
h.close()

In [ ]:
h = h5py.File('DeltaVparams_'+gamtag+'.hdf5', 'w')
dset = h.create_dataset('mean', data=np.array(DeltaVparams))
dset = h.create_dataset('jk_m0p2000', data=np.array(jk_DeltaVparams_m0p2000))
dset = h.create_dataset('jk_m0p3000', data=np.array(jk_DeltaVparams_m0p3000))
dset = h.create_dataset('jk_m0p4000', data=np.array(jk_DeltaVparams_m0p4000))
h.close()

In [ ]:
h = h5py.File('S3params_'+gamtag+'.hdf5', 'w')
dset = h.create_dataset('mean', data=np.array(S3params))
dset = h.create_dataset('jk_m0p2000', data=np.array(jk_S3params_m0p2000))
dset = h.create_dataset('jk_m0p3000', data=np.array(jk_S3params_m0p3000))
dset = h.create_dataset('jk_m0p4000', data=np.array(jk_S3params_m0p4000))
h.close()